This notebook runs ECMWF's aifs-single-v1 data-driven model, using ECMWF's [open data](https://www.ecmwf.int/en/forecasts/datasets/open-data) dataset and the [anemoi-inference](https://anemoi-inference.readthedocs.io/en/latest/apis/level1.html) package.

# 1. Install Required Packages and Imports

In [ ]:
# Uncomment the lines below to install the required packages

!pip install -q anemoi-inference[huggingface]==0.4.9
!pip install -q earthkit-regrid==0.4.0 ecmwf-opendata 

zsh:1: no matches found: anemoi-inference[huggingface]==0.4.9


It gave me the error:

I had to do:
pip install "anemoi-inference[huggingface]==0.4.9"
# anemoi-models==0.4.0 is installed via pip install -e .

In [1]:
import datetime
from collections import defaultdict

import numpy as np
import earthkit.data as ekd
import earthkit.regrid as ekr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from ecmwf.opendata import Client as OpendataClient

# 2. Retrieve Initial Conditions from ECMWF Open Data




### List of parameters to retrieve form ECMWF open data

In [4]:
PARAM_SFC = ["10u", "10v", "2d", "2t", "msl", "skt", "sp", "tcw", "lsm", "z", "slor", "sdor"]
PARAM_SOIL =["vsw","sot"]
PARAM_PL = ["gh", "t", "u", "v", "w", "q"]
LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]
SOIL_LEVELS = [1,2]

### Select a date

In [5]:
DATE = OpendataClient().latest()
dates = [datetime.datetime(2025, 9, 2, 0, 0), datetime.datetime(2025, 9, 1, 0, 0)]

In [6]:
DATES_LIST = dates

In [7]:
print("Initial date is", DATE)

Initial date is 2025-09-02 00:00:00


### Get the data from the ECMWF Open Data API

In [8]:
def get_open_data(param, levelist=[]):
    fields = defaultdict(list)
    # Get the data for the current date and the previous date
    for date in dates:
        data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist)
        for f in data:
            # Open data is between -180 and 180, we need to shift it to 0-360
            assert f.to_numpy().shape == (721,1440)
            values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
            # Interpolate the data to from 0.25 to N320
            values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            # Add the values to the list
            name = f"{f.metadata('param')}_{f.metadata('levelist')}" if levelist else f.metadata("param")
            fields[name].append(values)

    # Create a single matrix for each parameter
    for param, values in fields.items():
        fields[param] = np.stack(values)

    return fields


In [9]:
from collections import defaultdict
import numpy as np

def get_open_data(param, dates, levelist=[]):
    """
    Fetch ECMWF Open Data for multiple dates and optional pressure levels.
    
    Returns a dict: {param_name: stacked_array_over_dates}
    """
    fields = defaultdict(list)

    for date in dates:
        data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist)
        for f in data:
            # Shift longitude from -180-180 to 0-360
            assert f.to_numpy().shape == (721,1440)
            values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
            # Interpolate to N320 grid
            values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            # Name includes level if levels were requested
            name = f"{f.metadata('param')}_{f.metadata('levelist')}" if levelist else f.metadata("param")
            fields[name].append(values)

    # Stack all arrays per parameter
    for param_name, values in fields.items():
        fields[param_name] = np.stack(values)

    return fields

# --- Create input_state for every date ---
def create_input_states(dates):
    input_states = []
    
    # Fetch all fields at once for all dates
    fields = {}
    fields.update(get_open_data(param=PARAM_SFC, dates=dates))
    soil = get_open_data(param=PARAM_SOIL, dates=dates, levelist=SOIL_LEVELS)
    mapping = {'sot_1': 'stl1', 'sot_2': 'stl2',
               'vsw_1': 'swvl1','vsw_2': 'swvl2'}
    for k, v in soil.items():
        fields[mapping[k]] = v
    fields.update(get_open_data(param=PARAM_PL, dates=dates, levelist=LEVELS))
    
    # Transform GH to Z
    for level in LEVELS:
        gh = fields.pop(f"gh_{level}")
        fields[f"z_{level}"] = gh * 9.80665

    # Now create one input_state per date
    for i, date in enumerate(dates):
        single_fields = {}
        for k, v in fields.items():
            # Slice the i-th timestep for this date
            single_fields[k] = v[i]
        input_states.append({"date": date, "fields": single_fields})
    
    return input_states

# Usage:
all_input_states = create_input_states(dates=DATES_LIST)


<multiple>:   0%|          | 0.00/7.38M [00:00<?, ?B/s]

<multiple>:   0%|          | 0.00/7.38M [00:00<?, ?B/s]

20250902000000-0h-oper-fc.grib2:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

20250901000000-0h-oper-fc.grib2:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

<multiple>:   0%|          | 0.00/54.5M [00:00<?, ?B/s]

<multiple>:   0%|          | 0.00/53.4M [00:00<?, ?B/s]

### Get Input Fields

In [10]:
fields = {}

#### Add the single levels fields

In [11]:
fields.update(get_open_data(param=PARAM_SFC))

TypeError: get_open_data() missing 1 required positional argument: 'dates'

In [ ]:
soil=get_open_data(param=PARAM_SOIL,levelist=SOIL_LEVELS)

20250902000000-0h-oper-fc.grib2:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

20250901000000-0h-oper-fc.grib2:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Soil parameters have been renamed since training this model, we need to rename to the original names

In [ ]:
mapping = {'sot_1': 'stl1', 'sot_2': 'stl2',
           'vsw_1': 'swvl1','vsw_2': 'swvl2'}
for k,v in soil.items():
    fields[mapping[k]]=v

#### Add the pressure levels fields

In [ ]:
fields.update(get_open_data(param=PARAM_PL, levelist=LEVELS))

<multiple>:   0%|          | 0.00/54.3M [00:00<?, ?B/s]

<multiple>:   0%|          | 0.00/54.5M [00:00<?, ?B/s]

#### Convert geopotential height into geopotential

In [ ]:
# Transform GH to Z
for level in LEVELS:
    gh = fields.pop(f"gh_{level}")
    fields[f"z_{level}"] = gh * 9.80665

### Create Initial State

In [ ]:
for keys in input_state["fields"].keys():
    print(keys, input_state["fields"][keys].shape)

10u (2, 542080)
10v (2, 542080)
2d (2, 542080)
2t (2, 542080)
msl (2, 542080)
skt (2, 542080)
sp (2, 542080)
tcw (2, 542080)
lsm (2, 542080)
z (2, 542080)
slor (2, 542080)
sdor (2, 542080)
swvl1 (2, 542080)
swvl2 (2, 542080)
stl1 (2, 542080)
stl2 (2, 542080)
t_1000 (2, 542080)
t_925 (2, 542080)
t_850 (2, 542080)
t_700 (2, 542080)
t_600 (2, 542080)
t_500 (2, 542080)
t_400 (2, 542080)
t_300 (2, 542080)
t_250 (2, 542080)
t_200 (2, 542080)
t_150 (2, 542080)
t_100 (2, 542080)
t_50 (2, 542080)
u_1000 (2, 542080)
u_925 (2, 542080)
u_850 (2, 542080)
u_700 (2, 542080)
u_600 (2, 542080)
u_500 (2, 542080)
u_400 (2, 542080)
u_300 (2, 542080)
u_250 (2, 542080)
u_200 (2, 542080)
u_150 (2, 542080)
u_100 (2, 542080)
u_50 (2, 542080)
v_1000 (2, 542080)
v_925 (2, 542080)
v_850 (2, 542080)
v_700 (2, 542080)
v_600 (2, 542080)
v_500 (2, 542080)
v_400 (2, 542080)
v_300 (2, 542080)
v_250 (2, 542080)
v_200 (2, 542080)
v_150 (2, 542080)
v_100 (2, 542080)
v_50 (2, 542080)
w_1000 (2, 542080)
w_925 (2, 542080)
w_

4. Our Dataset 

In [12]:
import json
import numpy as np
from bwdl.io.storage.read_write import ZarrOpener

ds_path = "gs://processed-era5_aifs-v0dot2_6h_n320_only_2021-plus-soil/test.zarr"
zarr_opener = ZarrOpener()
# select january month 
ds = zarr_opener.open_zarr(ds_path, consolidated=False).sel(time=slice("2021-09-01", "2021-09-03")).load()
print(ds.time.values)

multistep_input = 2

name_map = {
    # surface vars
    "10m_u_component_of_wind": "10u",
    "10m_v_component_of_wind": "10v",
    "2m_dewpoint_temperature": "2d",
    "2m_temperature": "2t",
    "mean_sea_level_pressure": "msl",
    "surface_pressure": "sp",
    "skin_temperature": "skt",
    "land_sea_mask": "lsm",
    "orography": "z",
    "standard_deviation_of_orography": "sdor",
    "slope_of_sub_gridscale_orography": "slor",
    "total_column_water": "tcw",
    "total_precipitation": "tp",   # si présent comme "tp"
    "convective_precipitation": "cp",
    # forcings
    "cos_latitude": "cos_latitude",
    "sin_latitude": "sin_latitude",
    "cos_longitude": "cos_longitude",
    "sin_longitude": "sin_longitude",
    "cos_julian_day": "cos_julian_day",
    "sin_julian_day": "sin_julian_day",
    "cos_local_time": "cos_local_time",
    "sin_local_time": "sin_local_time",
    "insolation": "insolation",
    # pressure-level vars (need suffix)
    "geopotential": "z",
    "temperature": "t",
    "specific_humidity": "q",
    "u_component_of_wind": "u",
    "v_component_of_wind": "v",
    "vertical_velocity": "w",
    # soil vars
    "soil_temperature_level_1": "stl1",
    "soil_temperature_level_2": "stl2",
    "volumetric_soil_water_layer_1": "swvl1",
    "volumetric_soil_water_layer_2": "swvl2",
}


def dataset_to_multistep_dict(ds, start_idx, multistep_input, date_coord="time"):
    """Convertit un bloc multistep d'un Dataset en dict {date, fields}"""
    date = str(ds[date_coord].isel(time=start_idx).values)  
    fields = {}

    for var in ds.data_vars:
        if "pressure_level" in ds[var].dims:
            for lev_val in ds["pressure_level"].values:
                short_name = name_map[var]
                key = f"{short_name}_{lev_val}"
                arr = ds[var].isel(
                    time=slice(start_idx, start_idx + multistep_input)).sel(
                    pressure_level=lev_val).values
                arr = arr.reshape(multistep_input, -1)  
                fields[key] = arr.tolist()
        else:
            short_name = name_map.get(var, var)
            if "time" not in ds[var].dims:
                arr = ds[var].values
                arr = np.expand_dims(arr, axis=0)
                arr = np.repeat(arr, multistep_input, axis=0)
                fields[short_name] = arr.tolist()
            else:
                arr = ds[var].isel(
                    time=slice(start_idx, start_idx + multistep_input)
                ).values
                arr = arr.reshape(multistep_input, -1)
                fields[short_name] = arr.tolist()

    return {"date": date, "fields": fields}


all_dicts = []

for start_idx in range(ds.dims["time"] - multistep_input + 1):
    print(start_idx)
    one_dict = dataset_to_multistep_dict(ds, start_idx, multistep_input)
    for k, v in one_dict["fields"].items():
        one_dict["fields"][k] = np.array(v)
    one_dict["date"] = datetime.datetime.fromisoformat(one_dict["date"].replace("Z", "+00:00"))
    
    all_dicts.append(one_dict)

['2021-09-01T00:00:00.000000000' '2021-09-01T06:00:00.000000000'
 '2021-09-01T12:00:00.000000000' '2021-09-01T18:00:00.000000000'
 '2021-09-02T00:00:00.000000000' '2021-09-02T06:00:00.000000000'
 '2021-09-02T12:00:00.000000000' '2021-09-02T18:00:00.000000000'
 '2021-09-03T00:00:00.000000000' '2021-09-03T06:00:00.000000000'
 '2021-09-03T12:00:00.000000000' '2021-09-03T18:00:00.000000000']
0


/tmp/ipykernel_217310/2033328958.py:88: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for start_idx in range(ds.dims["time"] - multistep_input + 1):


1
2
3
4
5
6
7
8
9
10


In [13]:
import numpy as np
import datetime

import numpy as np

# --- helper pour stats ---
def compute_stats(arr):
    return {
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
    }

# --- agrégation stats sur les premières 5 dates de all_dicts ---
stats_all_dicts = {}
for d in all_dicts[:5]:  # prendre les 5 premières entrées
    for k, v in d["fields"].items():
        arr = np.array(v)
        stats_all_dicts.setdefault(k, []).append(arr)

# stack par variable → (5, ...)
final_stats_all_dicts = {k: compute_stats(np.stack(v_list, axis=0))
                         for k, v_list in stats_all_dicts.items()}

# --- stats pour all_input_states ---
# Ici on prend toutes les dates dans all_input_states
stats_all_input_states = {}
for k in all_input_states[0]["fields"].keys():
    # empile les valeurs pour toutes les dates
    stacked = np.stack([state["fields"][k] for state in all_input_states], axis=0)
    stats_all_input_states[k] = compute_stats(stacked)

# --- print results ---
print(f"{'Variable':<20} | {'All_dicts (5 dates)':<55} | {'All_input_states':<55}")
print("-" * 135)

for k in sorted(set(final_stats_all_dicts.keys()) | set(stats_all_input_states.keys())):
    s_all = final_stats_all_dicts.get(k, {})
    s_inp = stats_all_input_states.get(k, {})

    all_str = f"mean={s_all.get('mean', 'NA'):.3f}, std={s_all.get('std', 'NA'):.3f}, min={s_all.get('min', 'NA'):.3f}, max={s_all.get('max', 'NA'):.3f}" if s_all else "NA"
    inp_str = f"mean={s_inp.get('mean', 'NA'):.3f}, std={s_inp.get('std', 'NA'):.3f}, min={s_inp.get('min', 'NA'):.3f}, max={s_inp.get('max', 'NA'):.3f}" if s_inp else "NA"

    print(f"{k:<20} | {all_str:<55} | {inp_str:<55}")
# import pandas as pd

# # --- build rows for DataFrame ---
# rows = []
# for k in sorted(set(final_stats_all_dicts.keys()) | set(stats_input_state.keys())):
#     s_all = final_stats_all_dicts.get(k, {})
#     s_inp = stats_input_state.get(k, {})
    
#     rows.append({
#         "Variable": k,
#         "All_mean": s_all.get("mean", None),
#         "All_std": s_all.get("std", None),
#         "All_min": s_all.get("min", None),
#         "All_max": s_all.get("max", None),
#         "Input_mean": s_inp.get("mean", None),
#         "Input_std": s_inp.get("std", None),
#         "Input_min": s_inp.get("min", None),
#         "Input_max": s_inp.get("max", None),
#     })

# # --- create DataFrame ---
# stats_df = pd.DataFrame(rows)

# # --- pretty print ---
# print(stats_df.to_string(index=False))

# # --- optional: save to CSV ---
# stats_df.to_csv("stats_summary.csv", index=False)



Variable             | All_dicts (5 dates)                                     | All_input_states                                       
---------------------------------------------------------------------------------------------------------------------------------------
10u                  | mean=-0.340, std=5.545, min=-25.194, max=28.058         | mean=-0.032, std=5.491, min=-25.033, max=25.674        
10v                  | mean=0.683, std=4.537, min=-23.018, max=20.690          | mean=0.705, std=4.688, min=-23.325, max=25.026         
2d                   | mean=284.042, std=14.017, min=197.259, max=304.553      | mean=284.004, std=14.104, min=201.414, max=303.873     
2t                   | mean=289.263, std=14.405, min=200.956, max=321.330      | mean=289.017, std=14.438, min=201.414, max=317.452     
cos_julian_day       | mean=-0.498, std=0.006, min=-0.507, max=-0.489          | NA                                                     
cos_latitude         | mean=0.782, std=0.2

In [20]:
# --- similarity test helper ---
def similar_distributions(s_all, s_inp, rel_tol=0.15, abs_tol=1e-2):
    if not s_all or not s_inp:
        return False
    mean_all, mean_inp = s_all["mean"], s_inp["mean"]
    std_all, std_inp = s_all["std"], s_inp["std"]

    mean_close = abs(mean_all - mean_inp) <= abs_tol or abs(mean_all - mean_inp) / (abs(mean_all) + 1e-9) < rel_tol
    std_close  = abs(std_all  - std_inp)  <= abs_tol or abs(std_all  - std_inp)  / (abs(std_all)  + 1e-9) < rel_tol

    return mean_close and std_close


# --- couleurs ---
GREEN = "\033[92m"
RED = "\033[91m"
RESET = "\033[0m"

# --- print results with colored marks ---
print(f"{'Variable':<20} | {'Similarity'}")
print("-" * 35)

for k in sorted(set(final_stats_all_dicts.keys()) | set(stats_all_input_states.keys())):
    s_all = final_stats_all_dicts.get(k, {})
    s_inp = stats_all_input_states.get(k, {})

    ok = similar_distributions(s_all, s_inp)
    mark = f"{GREEN}✔{RESET}" if ok else f"{RED}✘{RESET}"
    print(f"{k:<20} | {mark}")

Variable             | Similarity
-----------------------------------
10u                  | ✘
10v                  | ✔
2d                   | ✔
2t                   | ✔
cos_julian_day       | ✘
cos_latitude         | ✘
cos_local_time       | ✘
cos_longitude        | ✘
insolation           | ✘
lsm                  | ✔
msl                  | ✔
q_100                | ✔
q_1000               | ✔
q_150                | ✔
q_200                | ✔
q_250                | ✔
q_300                | ✔
q_400                | ✔
q_50                 | ✔
q_500                | ✔
q_600                | ✔
q_700                | ✔
q_850                | ✔
q_925                | ✔
sdor                 | ✘
sin_julian_day       | ✘
sin_latitude         | ✘
sin_local_time       | ✘
sin_longitude        | ✘
skt                  | ✔
slor                 | ✔
sp                   | ✔
stl1                 | ✔
stl2                 | ✔
swvl1                | ✔
swvl2                | ✔
t_100                | ✔
t_100

In [22]:
for k in sorted(set(final_stats_all_dicts.keys()) | set(stats_all_input_states.keys())):
    s_all = final_stats_all_dicts.get(k, {})
    s_inp = stats_all_input_states.get(k, {})
    if not similar_distributions(s_all, s_inp):
        print(f"Difference in {k}: All_dicts stats {s_all}, Input_states stats {s_inp}")
        

Difference in 10u: All_dicts stats {'mean': -0.3398540976928567, 'std': 5.54534756856827, 'min': -25.19403076171875, 'max': 28.05780029296875}, Input_states stats {'mean': -0.03249139139297582, 'std': 5.490750066195517, 'min': -25.032923623957853, 'max': 25.67402937344243}
Difference in cos_julian_day: All_dicts stats {'mean': -0.49812626242637636, 'std': 0.005593505677552589, 'min': -0.5074302554130554, 'max': -0.48878538608551025}, Input_states stats {}
Difference in cos_latitude: All_dicts stats {'mean': 0.7816197667462891, 'std': 0.2252909996057824, 'min': 0.0037545973900705576, 'max': 0.9999970197677612}, Input_states stats {}
Difference in cos_local_time: All_dicts stats {'mean': -3.025320841278771e-12, 'std': 0.7071067805279628, 'min': -1.0, 'max': 1.0}, Input_states stats {}
Difference in cos_longitude: All_dicts stats {'mean': -3.025320841278771e-11, 'std': 0.7071067804772548, 'min': -1.0, 'max': 1.0}, Input_states stats {}
Difference in insolation: All_dicts stats {'mean': 0.

In [23]:
# --- header ---
print(f"{'Variable':<20} | {'All_dicts (5 dates)':<55} | {'All_input_states':<55}")
print("-" * 135)

for k in sorted(set(final_stats_all_dicts.keys()) | set(stats_all_input_states.keys())):
    s_all = final_stats_all_dicts.get(k, {})
    s_inp = stats_all_input_states.get(k, {})

    # ne garder que les variables qui diffèrent
    if not similar_distributions(s_all, s_inp):
        all_str = f"mean={s_all.get('mean', 'NA'):.3f}, std={s_all.get('std', 'NA'):.3f}, min={s_all.get('min', 'NA'):.3f}, max={s_all.get('max', 'NA'):.3f}" if s_all else "NA"
        inp_str = f"mean={s_inp.get('mean', 'NA'):.3f}, std={s_inp.get('std', 'NA'):.3f}, min={s_inp.get('min', 'NA'):.3f}, max={s_inp.get('max', 'NA'):.3f}" if s_inp else "NA"

        print(f"{k:<20} | {all_str:<55} | {inp_str:<55}")

Variable             | All_dicts (5 dates)                                     | All_input_states                                       
---------------------------------------------------------------------------------------------------------------------------------------
10u                  | mean=-0.340, std=5.545, min=-25.194, max=28.058         | mean=-0.032, std=5.491, min=-25.033, max=25.674        
cos_julian_day       | mean=-0.498, std=0.006, min=-0.507, max=-0.489          | NA                                                     
cos_latitude         | mean=0.782, std=0.225, min=0.004, max=1.000             | NA                                                     
cos_local_time       | mean=-0.000, std=0.707, min=-1.000, max=1.000           | NA                                                     
cos_longitude        | mean=-0.000, std=0.707, min=-1.000, max=1.000           | NA                                                     
insolation           | mean=0.249, std=0.3

In [ ]:
input_state["fields"].keys()

dict_keys(['10u', '10v', '2d', '2t', 'msl', 'skt', 'sp', 'tcw', 'lsm', 'z', 'slor', 'sdor', 'swvl1', 'swvl2', 'stl1', 'stl2', 't_1000', 't_925', 't_850', 't_700', 't_600', 't_500', 't_400', 't_300', 't_250', 't_200', 't_150', 't_100', 't_50', 'u_1000', 'u_925', 'u_850', 'u_700', 'u_600', 'u_500', 'u_400', 'u_300', 'u_250', 'u_200', 'u_150', 'u_100', 'u_50', 'v_1000', 'v_925', 'v_850', 'v_700', 'v_600', 'v_500', 'v_400', 'v_300', 'v_250', 'v_200', 'v_150', 'v_100', 'v_50', 'w_1000', 'w_925', 'w_850', 'w_700', 'w_600', 'w_500', 'w_400', 'w_300', 'w_250', 'w_200', 'w_150', 'w_100', 'w_50', 'q_1000', 'q_925', 'q_850', 'q_700', 'q_600', 'q_500', 'q_400', 'q_300', 'q_250', 'q_200', 'q_150', 'q_100', 'q_50', 'z_1000', 'z_925', 'z_850', 'z_700', 'z_600', 'z_500', 'z_400', 'z_300', 'z_250', 'z_200', 'z_150', 'z_100', 'z_50'])

In [ ]:
dict = all_dicts[0]
# transform lists to arrays
for k, v in dict["fields"].items():
    dict["fields"][k] = np.array(v)
    

In [ ]:
dict["date"] = datetime.datetime.fromisoformat(dict["date"].replace("Z", "+00:00"))


TypeError: 'str' object cannot be interpreted as an integer

In [ ]:
input_state_bw = dict

In [ ]:
input_state_bw

{'date': datetime.datetime(2021, 1, 1, 0, 0),
 'fields': {'10v': array([[ 4.23976135,  5.88819885,  6.81983948, ...,  3.06007385,
           2.30226135,  1.21339417],
         [ 4.47065735,  6.76948547,  8.20698547, ...,  2.38569641,
           1.21577454, -0.15922546]]),
  '10u': array([[-5.93379211, -4.08808899, -1.84785461, ..., -1.80879211,
          -2.80293274, -3.49824524],
         [-7.95797729, -5.88375854, -3.17575073, ..., -2.77145386,
          -3.50875854, -3.80172729]]),
  'cos_latitude': array([[0.0037546, 0.0037546, 0.0037546, ..., 0.0037546, 0.0037546,
          0.0037546],
         [0.0037546, 0.0037546, 0.0037546, ..., 0.0037546, 0.0037546,
          0.0037546]]),
  'cos_julian_day': array([[1.        , 1.        , 1.        , ..., 1.        , 1.        ,
          1.        ],
         [0.99999076, 0.99999076, 0.99999076, ..., 0.99999076, 0.99999076,
          0.99999076]]),
  '2d': array([[262.64035034, 262.69503784, 262.66964722, ..., 238.36886597,
          238.3

# 3. Load the Model and Run the Forecast

### Download the Model's Checkpoint from Hugging Face & create a Runner

In [ ]:
checkpoint = {"huggingface":"ecmwf/aifs-single-1.0"}

To reduce the memory usage of the model certain environment variables can be set, like the number of chunks of the model's mapper.
Please refer to:
- https://anemoi.readthedocs.io/projects/models/en/latest/modules/layers.html#anemoi-inference-num-chunks
- https://pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf

for more information. To do so, you can use the code below:
```
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True' 
os.environ['ANEMOI_INFERENCE_NUM_CHUNKS']='16'
```

In [ ]:
runner = SimpleRunner(checkpoint, device="cuda")

**Note - changing the device from GPU to CPU**

- Running the transformer model used on the CPU is tricky, it depends on the FlashAttention library which only supports Nvidia and AMD GPUs, and is optimised for performance and memory usage
- In newer versions of anemoi-models, v0.4.2 and above, there is an option to switch off flash attention and uses Pytorchs Scaled Dot Product Attention (SDPA). The code snippet below shows how to overwrite a model from a checkpoint to use SDPA. Unfortunately it's not optimised for memory usage in the same way, leading to much greater memory usage. Please refer to https://github.com/ecmwf/anemoi-inference/issues/119 for more details 

#### Run the forecast

In [ ]:
input_state_bw["fields"]["z_500"].mean()

55317.12049144038

In [ ]:
for state in runner.run(input_state=input_state_bw, lead_time=6):
    print_state(state)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/ubuntu/miniconda3/envs/bwdl/lib/python3.11/site-packages/anemoi/utils/config.py:209: UserWarning: Modifying an instance of DotDict(). This class is intended to be immutable.
  warnings.warn("Modifying an instance of DotDict(). This class is intended to be immutable.")


> /home/ubuntu/miniconda3/envs/bwdl/lib/python3.11/site-packages/anemoi/models/interface/__init__.py(111)predict_step()
    109             import pdb
    110             pdb.set_trace()
--> 111             x = batch[:, 0 : self.multi_step, None, ...]  # add dummy ensemble dimension as 3rd index
    112 
    113             y_hat = self(x)

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
> /home/ubuntu/miniconda3/envs/bwdl/lib/python3.11/site-packages/anemoi/models/interface/__init__.py(102)predict_step()
    100         batch = self.pre_processors(batch, in_place=False)
    101 
--> 102         with torch.no_grad():
    103 
    104             assert (

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user

😀 date=2021-01-01T06:00:00 latitudes=(542080,) longitudes=(542080,) fields=102

    q_50   shape=(542080,) min=1.26885e-06    max=3.28137e-06   
    t_1000 shape=(542080,) min=226.903        max=318.292       
    v_925  shape=(542080,) min=-32.8898       max

: 

In [ ]:
state["fields"]["z_500"].mean()

**Note** 
Due to the non-determinism of GPUs, users will be unable to exactly reproduce an official AIFS forecast when running AIFS Single themselves.
If you want to enforece determinism at GPU level, you can do so enforcing the following settings:

```
#First in your terminal
export CUBLAS_WORKSPACE_CONFIG=:4096:8

#And then before running inference:
import torch
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

```
Using the above approach will significantly increase runtime. Additionally, the input conditions come from open data, which we reproject from o1280 (the original projection of IFS initial conditions) to n320 (AIFS resolution) by first converting them to a 0.25-degree grid. In the operational setup, however, data is reprojected directly from o1280 to n320. This difference in reprojection methods may lead to variations in the resulting input conditions, causing minor differences in the forecast.

# 4. Inspect the generated forecast

#### Plot a field

In [ ]:
# To be able to run the plotting section below you need to install additional dependencies

# !pip install -q matplotlib
# !pip install -q cartopy

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.tri as tri

In [ ]:
def fix(lons):
    # Shift the longitudes from 0-360 to -180-180
    return np.where(lons > 180, lons - 360, lons)

latitudes = state["latitudes"]
longitudes = state["longitudes"]
values = state["fields"]["100u"]

fig, ax = plt.subplots(figsize=(11, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=":")

triangulation = tri.Triangulation(fix(longitudes), latitudes)

contour=ax.tricontourf(triangulation, values, levels=20, transform=ccrs.PlateCarree(), cmap="RdBu")
cbar = fig.colorbar(contour, ax=ax, orientation="vertical", shrink=0.7, label="100u")

plt.title("100m winds (100u) at {}".format(state["date"]))
plt.show()